In [4]:
filename = 'brenda_2023_1.txt'
keyword = 'TN' # TURNOVER_NUMBER
#keyword = 'KM' # KM-value
# keyword = 'KI' # KI-value

fout = open(keyword + '.txt', 'w')
if keyword == 'KI':
    title = 'EC-number'+'\t'+'Organism'+'\t'+'Inhibitor'+'\t'+'Value'+'\t'+'Commentary'+'\t'+'Reference'+'\t'+'Protein-ID'+'\n'
else:
    title = 'EC-number'+'\t'+'Organism'+'\t'+'Substrate'+'\t'+'Value'+'\t'+'Commentary'+'\t'+'Reference'+'\t'+'Protein-ID'+'\n'
fout.write(title)
fout.close()

import re

def processOrganism(org_id_tmp): # Note: EMBL not considered
    if org_id_tmp.lower().find('swissprot') != -1: # SwissProt
        if org_id_tmp.lower().find(' and ', 0, org_id_tmp.lower().find('swissprot')) != -1:
            organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find(' and ')].strip().rfind(' ')].strip()
        else:
            organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find('swissprot')].strip().rfind(' ')].strip()
        proteinID = org_id_tmp.replace(organism,'')[:org_id_tmp.replace(organism,'').lower().find('swissprot')].strip() + ' SwissProt'
    elif org_id_tmp.lower().find('uniprot') != -1: # UniProt
        if org_id_tmp.lower().find(' and ', 0, org_id_tmp.lower().find('uniprot')) != -1:
            organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find(' and ')].strip().rfind(' ')].strip()
        else:
            organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find('uniprot')].strip().rfind(' ')].strip()
        proteinID = org_id_tmp.replace(organism,'')[:org_id_tmp.replace(organism,'').lower().find('uniprot')].strip() + ' UniProt'
    elif org_id_tmp.lower().find('trembl') != -1: # TrEMBL
        if org_id_tmp.lower().find(' and ', 0, org_id_tmp.lower().find('trembl')) != -1:
            organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find(' and ')].strip().rfind(' ')].strip()
        else:
            organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find('trembl')].strip().rfind(' ')].strip()
        proteinID = org_id_tmp.replace(organism,'')[:org_id_tmp.replace(organism,'').lower().find('trembl')].strip() + ' TrEMBL'
    elif org_id_tmp.lower().find('genbank') != -1: # GenBank
        organism = org_id_tmp[:org_id_tmp[:org_id_tmp.lower().find('genbank')].strip().rfind(' ')].strip()
        proteinID = org_id_tmp.replace(organism,'')[:org_id_tmp.replace(organism,'').lower().find('genbank')].strip() + ' GenBank'
    else:
        organism = org_id_tmp
        proteinID = ''
    return organism, proteinID

fhand = open(filename)
line_tmp = ''
for line in fhand:
    if line.split('\t')[0] == '': # the current line starts with TAB
        if line_tmp.split('\t')[0] in ['ID', 'PR', keyword]: # the previous line starts with 'ID', 'PR' or keyword
            line_tmp = line_tmp.rstrip() + line.replace('	',' ')
        else:
            line_tmp = ''
    else:
        if line_tmp.split('\t')[0] in ['ID', 'PR', keyword]: # the previous line starts with 'ID', 'PR' or keyword
            if line_tmp.split('\t')[0] == 'ID':
                ec_id = line_tmp.split('\t')[1].rstrip()
                organism_dict = {}
            elif line_tmp.split('\t')[0] == 'PR':
                org_num = re.findall('#\d+#', line_tmp.split('\t')[1].rstrip())[0]      #split分成PR和剩下的部分；使用正则表达式 #\d+# 查找#数字#的模式，其中\d+表示匹配一个或多个数字。
                org_id = line_tmp.replace('PR	'+org_num,'').split('   ')[0].strip()   #查看数据集知：split三个空格已经删除了文献信息
                if org_id.rfind(' <') != -1:                                            #可能多此一举了，目前没发现经过上一行之后还有<的情况
                    org_id = org_id[:org_id.rfind(' <')].strip()                        
                organism_dict[org_num] = org_id
                #print(org_id)
            else: # keyword:
                fout = open('files/' + keyword + '.txt', 'a')
                
                info_tmp = line_tmp.split('\t')[1]
                value = re.findall('# .+ {', info_tmp)[0][2:-2]
                substrate = re.findall(' {.+} ', info_tmp)[0][2:-2]
                
                if info_tmp.find('(',info_tmp.find('} ')) != -1: # if there is commentary
                    # get commentary position
                    cmt_tmp = info_tmp[info_tmp.find('} ')+2:info_tmp.rfind(' ')].strip()
                    cmt_tmp = cmt_tmp[2:-1].split('; #')
                    for i in cmt_tmp:
                        i = '#'+i
                        #print(i)
                        cmt = i[i.find(' ')+1:i.rfind(' ')]
                        ref = i[i.rfind(' '):]
                        org_tmp = i[:i.find(' ')]
                        #print(org_tmp)
                        #print(cmt_tmp)
                        if org_tmp.find(',') != -1: # the case like "#20,21#"
                            for j in org_tmp.replace('#','').split(','):
                                org_num_tmp = '#'+j+'#'
                                org_id_tmp = organism_dict[org_num_tmp]
                                organism, proteinID = processOrganism(org_id_tmp)
                                line_to_write = ec_id+'\t'+organism+'\t'+substrate+'\t'+value+'\t'+cmt+'\t'+ref+'\t'+proteinID+'\n'
                                fout.write(line_to_write)
                        else:
                            org_id_tmp = organism_dict[org_tmp]
                            organism, proteinID = processOrganism(org_id_tmp)
                            line_to_write = ec_id+'\t'+organism+'\t'+substrate+'\t'+value+'\t'+cmt+'\t'+ref+'\t'+proteinID+'\n'
                            fout.write(line_to_write)
                    
                else: # if there is no commentary
                    cmt = ''
                    ref = re.findall(' <.+>', info_tmp)[0]
                    org_tmp = info_tmp[:info_tmp.find(' ')]
                    if org_tmp.find(',') != -1: # the case like "#154,155,156#"
                        for j in org_tmp.replace('#','').split(','):
                            org_num_tmp = '#'+j+'#'
                            org_id_tmp = organism_dict[org_num_tmp]
                            organism, proteinID = processOrganism(org_id_tmp)
                            line_to_write = ec_id+'\t'+organism+'\t'+substrate+'\t'+value+'\t'+cmt+'\t'+ref+'\t'+proteinID+'\n'
                            fout.write(line_to_write)
                    else:
                        org_id_tmp = organism_dict[org_tmp]
                        organism, proteinID = processOrganism(org_id_tmp)
                        line_to_write = ec_id+'\t'+organism+'\t'+substrate+'\t'+value+'\t'+cmt+'\t'+ref+'\t'+proteinID+'\n'
                        fout.write(line_to_write)
                fout.close()
                
            line_tmp = line
        else:
            line_tmp = line
    
fhand.close()
